# How well calibrated are NBA win probabilities? A look at Basketlog's predictions

This notebook checks the [Basketlog](https://basketlog-app.com/en) NBA model against real results: every regular-season game of 2025-26 and the 2026-27 season so far.

Each probability was computed **before tip-off, using only results known at the time**, so there is no look-ahead. The question is simple: when the model says 70%, does the favorite win about 70% of the time?

Data: [NBA game predictions and results (Basketlog)](https://www.kaggle.com/datasets/basketlogcontact/nba-game-predictions-and-results-basketlog), updated daily, CC BY 4.0. Live version of these numbers: [basketlog-app.com/en/app/bilan](https://basketlog-app.com/en/app/bilan).

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

files = sorted(glob.glob("/kaggle/input/**/nba-predictions-*.csv", recursive=True))
frames = []
for f in files:
    d = pd.read_csv(f)
    d["season"] = f.split("nba-predictions-")[-1].replace(".csv", "")
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
df["home_won"] = (df["home_score"] > df["away_score"]).astype(int)
df["fav_prob"] = df["predicted_winner_probability"]
df["correct"] = df["correct"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)
print(files)
df.groupby("season").agg(games=("game_id", "count"), accuracy=("correct", "mean")).round(3)

## Accuracy by season

The favorite's win rate is the simplest check. In the NBA, a good model usually lands in the high 60s: upsets are frequent, so nothing gets close to 100%.

## Calibration

Group games by the stated probability of the home team and compare with how often the home team actually won. A perfectly calibrated model sits on the diagonal.

In [ ]:
bins = np.linspace(0, 1, 11)
df["bucket"] = pd.cut(df["home_win_probability"], bins, include_lowest=True)
cal = df.groupby("bucket", observed=True).agg(stated=("home_win_probability", "mean"), actual=("home_won", "mean"), games=("game_id", "count"))
print(cal.round(3))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfect calibration")
ax.scatter(cal["stated"], cal["actual"], s=cal["games"] * 1.5, color="#e8640c", alpha=0.8, label="Basketlog (size = games)")
ax.set_xlabel("Stated home win probability")
ax.set_ylabel("Actual home win rate")
ax.set_title("Basketlog NBA model calibration")
ax.legend()
plt.show()

## Brier score

The Brier score is the mean squared error of the probabilities (0 is perfect, 0.25 is a coin flip). It rewards being confident only when it is deserved.

In [ ]:
brier = ((df["home_win_probability"] - df["home_won"]) ** 2).groupby(df["season"]).mean()
coin = pd.Series(0.25, index=brier.index)
pd.DataFrame({"brier": brier.round(4), "coin_flip": coin})

## Accuracy by confidence level

Strong favorites should win much more often than slight ones.

In [ ]:
df["confidence"] = pd.cut(df["fav_prob"], [0.5, 0.6, 0.7, 0.8, 0.9, 1.0], include_lowest=True)
df.groupby("confidence", observed=True).agg(games=("game_id", "count"), favorite_won=("correct", "mean"), avg_stated=("fav_prob", "mean")).round(3)

## Projected score error

The model also projects a final score. How far off is the projected margin, on average?

In [ ]:
proj_margin = df["predicted_home_points"] - df["predicted_away_points"]
real_margin = df["home_score"] - df["away_score"]
print("Mean absolute margin error:", round((proj_margin - real_margin).abs().mean(), 1), "points")
print("Std of real margins:", round(real_margin.std(), 1), "points")

## Takeaways

* The favorite wins roughly two thirds to seven in ten games, which is about as good as it gets in the NBA.
* The calibration points stay close to the diagonal: a 70% favorite really does lose about 3 games in 10.
* Single games stay noisy (see the margin error above), which is why a probability says more than a simple pick.

Daily probabilities for every NBA and EuroLeague game, playoff odds and the 2027 draft lottery odds are free at [basketlog-app.com/en](https://basketlog-app.com/en). A free JSON API is documented at [basketlog-app.com/en/app/donnees](https://basketlog-app.com/en/app/donnees).